# Поглавље 7: Изградња апликација за чет
## Брзи почетак са OpenAI API-јем

Ова свеска је прилагођена из [Azure OpenAI Samples Repository](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) која укључује свеске које приступају [Azure OpenAI](aoai-assignment.ipynb) услугама.

Python OpenAI API ради и са Azure OpenAI моделима, уз неке измене. Сазнајте више о разликама овде: [Како пребацити између OpenAI и Azure OpenAI крајњих тачака помоћу Питхона](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)


# Преглед  
"Велики језички модели су функције које мапирају текст на текст. Датим улазним стрингом текста, велики језички модел покушава да предвиди који текст ће уследити"(1). Овај „брзи почетак“ нотебук ће корисницима представити концепте високог нивоа LLM, основне захтеве пакета за почетак рада са AML, благи увод у дизајн упита, као и неколико кратких примера различитих случајева употребе. 


## Садржај  

[Преглед](#overview)  
[Како користити OpenAI Услугу](#how-to-use-openai-service)  
[1. Креирање ваше OpenAI Услуге](#1.-creating-your-openai-service)  
[2. Инсталација](#2.-installation)    
[3. Акредитације](#3.-credentials)  

[Примери коришћења](#use-cases)    
[1. Сажми текст](#1.-summarize-text)  
[2. Класификуј текст](#2.-classify-text)  
[3. Генериши нове називе производа](#3.-generate-new-product-names)  
[4. Фино намести класификатор](#4.fine-tune-a-classifier)  

[Референце](#references)


### Направите свој први упит  
Ова кратка вежба ће пружити основни увод у слање упита моделу OpenAI за једноставан задатак „сумаризација“.


**Кораци**:  
1. Инсталирајте OpenAI библиотеку у вашем Python окружењу  
2. Учитајте стандардне помоћне библиотеке и подесите уобичајене безбедносне акредитиве за OpenAI услугу коју сте креирали  
3. Изаберите модел за ваш задатак  
4. Креирајте једноставан упит за модел  
5. Пошаљите захтев API-ју модела!


### 1. Инсталирајте OpenAI


In [ ]:
%pip install openai python-dotenv

### 2. Увезите помоћне библиотеке и креирајте креденцијале


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### 3. Проналажење правог модела  
Модели попут GPT-4o и GPT-4o мини могу да разумеју и генеришу природни језик, и доступни су на OpenAI платформи са различитим нивоима снаге и брзине погодним за различите задатке.


In [ ]:
# Select a general purpose chat model
model = "gpt-5-mini"


## 4. Дизајн упита  

"Магија великих језичких модела је у томе што, пошто су обучени да минимизирају ову грешку предвиђања преко огромних количина текста, модели на крају уче концепте корисне за ова предвиђања. На пример, они уче концепте као што су"(1):

* како се пише
* како функционише граматика
* како се препрoизводи
* како се одговара на питања
* како се води разговор
* како се пише на многим језицима
* како се кодује
* итд.

#### Како контролисати велики језички модел  
"Од свих улаза у велики језички модел, највише утицаја има текстуални упит(1).

Веће језичке моделе можете упућивати да генеришу излаз на неколико начина:

Упутство: Реците моделу шта желите
Завршетак: Наведите модел да доврши почетак онога што желите
Демонстрација: Покажите моделу шта желите, са:
Неколико примера у упиту
Стотинама или хиљадама примера у скупу података за фино подешавање"



#### Постоје три основна правила за креирање упита:

**Покажи и реци**. Јасно назначите шта желите било кроз упутства, примере, или комбинацију оба. Ако желите да модел рангира листу ставки по азбучном реду или да класификује пасус по сентименту, покажите му да то желите.

**Обезбедите квалитетне податке**. Ако покушавате да направите класификатор или да натерате модел да прати образац, уверите се да има довољно примера. Обавезно проверите своје примере — модел је обично довољно паметан да уочи основне правописне грешке и да вам одговори, али такође може претпоставити да је то намерно и то може утицати на одговор.

**Проверите своја подешавања.** Параметри temperature и top_p контролишу колико је модел детерминистички у генерисању одговора. Ако тражите одговор где постоји само један тачан одговор, онда желите да их подесите ниже. Ако тражите разноврсније одговоре, онда их можда желите подесити више. Најчешћа грешка коју људи праве са овим подешавањима јесте да претпостављају да су то контроле "паметности" или "креативности".


Извор: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. Потврди!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### Поновите исти позив, како се резултати упоређују?


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## Сумирање текста  
#### Изазов  
Сумирајте текст додавањем 'tl;dr:' на крај текста. Приметите како модел разуме како да изведе низ задатака без додатних упутстава. Можете експериментисати са описнијим захтевима од tl;dr како бисте изменили понашање модела и прилагодили добијено сумирање(3).  

Недавни рад је показао значајне добитке у многим НЛП задацима и бенчмарковима преттренирањем на великом корпусу текста, а затим дофиношљавањем на одређеном задатку. Иако је архитектура обично независна од задатака, овај метод и даље захтева скупове за дофиношљавање специфичне за задатак са хиљадама или десетинама хиљада примера. Насупрот томе, људи генерално могу изводити нови језички задатак са само неколико примера или једноставним упутствима - нешто са чим се тренутни НЛП системи још увек углавном муче. Овде показујемо да значајно повећање језичких модела у великој мери побољшава независне од задатка перформансе са мало примера, понекад чак достижући конкурентност са досадашњим најбољим методама дофиношљавања. 



tl;dr  


# Вежбе за више случајева коришћења  
1. Сажети текст  
2. Класифицирати текст  
3. Генерисати нове називе производа


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Класификујте текст  
#### Изазов  
Класификујте ставке у категорије дате у тренутку инференције. У следећем примеру дајемо и категорије и текст за класификовање у упутству (*playground_reference). 

Упит купца: Здраво, један тастер на тастатури мог лаптопа се недавно покварио и потребна ми је замена:

Класификована категорија:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Генеришите нове називе производа
#### Изазов
Креирајте називе производа на основу примера речи. Овде укључујемо у упит информације о производу за који ћемо генерисати називе. Такође дајемо сличан пример да покажемо образац који желимо да добијемо. Поставили смо и вредност температуре високо да повећамо случајност и добијемо иновативније одговоре.

Опис производа: Уређај за прављење млечних шејкова за кућну употребу
Почетне речи: брз, здрав, компактни.
Називи производа: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Опис производа: Пар ципела који може да одговара на било коју величину стопала.
Почетне речи: прилагодљив, одговарајући, свестрани.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# Референце  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry портал](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Најбоље праксе за фино подешавање GPT модела за класификацију текста](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# За више помоћи  
[Тим за комерцијализацију OpenAI-а](AzureOpenAITeam@microsoft.com) 


# Доприносиоци
* Louis Li  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Изјава о одрицању одговорности**:
Овај документ је преведен коришћењем услуге за аутоматски превод [Co-op Translator](https://github.com/Azure/co-op-translator). Иако тежимо тачности, имајте у виду да аутоматски преводи могу садржати грешке или нетачности. Оригинални документ на његовом изворном језику треба сматрати ауторитативним извором. За критичне информације препоручује се професионални људски превод. Нисмо одговорни за било каква неспоразума или погрешна тумачења која произилазе из коришћења овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
